# P-Task8: Banking System

**Problem:** `BankAccount` class banani hai jo owner ka naam aur balance safely store kare. Balance class ke
bahar se access nahi hona chahiye. Deposit aur withdraw sirf validated methods se hon. Account print karne par
saaf summary aaye.

**Concepts used:** class, object, `__init__`, `self`, class vs instance attributes, `@classmethod`,
`@staticmethod`, public / `_protected` / `__private`, `@property` + `@setter`, `__str__`, `__repr__`.

## Access levels (naming convention)
| Naam | Matlab |
|---|---|
| `owner` | public, koi bhi use kar sakta hai |
| `_account_no` | protected, "andar ki cheez hai, bahar se mat chhedo" (sirf convention) |
| `__balance` | private, Python naam badal deta hai (name mangling) isliye bahar se seedha nahi milta |

In [1]:
class InsufficientFundsError(Exception):
    pass


class BankAccount:
    bank_name = "Smart District Bank"      # class attribute (sab accounts ke liye ek hi)
    total_accounts = 0
    MIN_DEPOSIT = 100

    def __init__(self, owner, opening_balance=0):
        self.owner = owner                 # setter se guzray ga
        self._account_no = 1000 + BankAccount.total_accounts + 1
        self.__balance = 0                 # private
        self.__history = []                # private
        BankAccount.total_accounts += 1
        if opening_balance > 0:
            self.deposit(opening_balance)

    # ---- owner: property + setter ----
    @property
    def owner(self):
        return self._owner

    @owner.setter
    def owner(self, value):
        if not isinstance(value, str) or len(value.strip()) < 3:
            raise ValueError("owner ka naam kam az kam 3 huroof ka hona chahiye")
        self._owner = value.strip().title()

    @property
    def account_no(self):
        return self._account_no

    # ---- static / class methods ----
    @staticmethod
    def is_valid_amount(amount):
        return isinstance(amount, (int, float)) and not isinstance(amount, bool) and amount > 0

    @classmethod
    def from_string(cls, text):
        """'sana,5000' jaisi string se account banata hai."""
        name, amount = text.split(",")
        return cls(name, float(amount))

    # ---- validated money methods ----
    def deposit(self, amount):
        if not self.is_valid_amount(amount):
            raise ValueError(f"deposit amount galat hai: {amount!r}")
        if amount < self.MIN_DEPOSIT:
            raise ValueError(f"minimum deposit Rs {self.MIN_DEPOSIT} hai")
        self.__balance += amount
        self.__history.append(("deposit", amount, self.__balance))
        return self.__balance

    def withdraw(self, amount):
        if not self.is_valid_amount(amount):
            raise ValueError(f"withdraw amount galat hai: {amount!r}")
        if amount > self.__balance:
            raise InsufficientFundsError(f"balance kam hai (maanga Rs {amount:,.0f}, hai Rs {self.__balance:,.0f})")
        self.__balance -= amount
        self.__history.append(("withdraw", amount, self.__balance))
        return self.__balance

    def statement(self):
        lines = [f"  {kind:<9} Rs {amount:>9,.2f}   balance: Rs {bal:>9,.2f}"
                 for kind, amount, bal in self.__history]
        return "\n".join(lines)

    # ---- printing ----
    def __str__(self):
        return (f"{self.bank_name}\n"
                f"  Account No : {self._account_no}\n"
                f"  Owner      : {self.owner}\n"
                f"  Balance    : Rs {self.__balance:,.2f}")

    def __repr__(self):
        return f"BankAccount(owner={self.owner!r}, account_no={self._account_no})"

## Account banana aur use karna

In [2]:
acc = BankAccount("ayesha khan", 5000)
print(acc)

Smart District Bank
  Account No : 1001
  Owner      : Ayesha Khan
  Balance    : Rs 5,000.00


In [3]:
acc.deposit(2500)
acc.withdraw(1200)
print(acc)
print()
print(acc.statement())

Smart District Bank
  Account No : 1001
  Owner      : Ayesha Khan
  Balance    : Rs 6,300.00

  deposit   Rs  5,000.00   balance: Rs  5,000.00
  deposit   Rs  2,500.00   balance: Rs  7,500.00
  withdraw  Rs  1,200.00   balance: Rs  6,300.00


## Validation check (galat cheezein)

In [4]:
tests = [
    ("deposit -500", lambda: acc.deposit(-500)),
    ("deposit 'abc'", lambda: acc.deposit("abc")),
    ("deposit 50 (minimum se kam)", lambda: acc.deposit(50)),
    ("withdraw 1,00,000", lambda: acc.withdraw(100000)),
    ("withdraw True", lambda: acc.withdraw(True)),
    ("naam 'ab'", lambda: BankAccount("ab")),
]
for label, action in tests:
    try:
        action()
    except (ValueError, InsufficientFundsError) as e:
        print(f"{label:<28} -> {type(e).__name__}: {e}")

deposit -500                 -> ValueError: deposit amount galat hai: -500
deposit 'abc'                -> ValueError: deposit amount galat hai: 'abc'
deposit 50 (minimum se kam)  -> ValueError: minimum deposit Rs 100 hai
withdraw 1,00,000            -> InsufficientFundsError: balance kam hai (maanga Rs 100,000, hai Rs 6,300)
withdraw True                -> ValueError: withdraw amount galat hai: True
naam 'ab'                    -> ValueError: owner ka naam kam az kam 3 huroof ka hona chahiye


## Balance bahar se access karne ki koshish

In [5]:
for attempt in ["acc.balance", "acc.__balance"]:
    try:
        eval(attempt)
    except AttributeError as e:
        print(f"{attempt:<15} -> AttributeError")

acc.balance     -> AttributeError
acc.__balance   -> AttributeError


Dono jagah error aaya. Note: Python mein `__balance` asal mein `_BankAccount__balance` ban jata hai,
yani agar koi jaan-boojh kar `acc._BankAccount__balance` likhe to mil jayega. Python "poori security" nahi deta,
sirf accident se bachata hai. Isliye is class mein balance ko kabhi direct set nahi kiya, sirf methods se.

## classmethod, class attribute aur `__repr__`

In [6]:
acc2 = BankAccount.from_string("bilal ahmed,12000")
acc3 = BankAccount("fatima noor")

print(repr(acc2))
print("Total accounts :", BankAccount.total_accounts)
print("Bank name      :", acc2.bank_name)
print()
print(acc3)

BankAccount(owner='Bilal Ahmed', account_no=1002)
Total accounts : 3
Bank name      : Smart District Bank

Smart District Bank
  Account No : 1003
  Owner      : Fatima Noor
  Balance    : Rs 0.00


## Property setter check

In [7]:
acc3.owner = "  fatima noor khan "
print(acc3.owner)

Fatima Noor Khan


## Conclusion
Balance `__balance` private hai aur sirf `deposit()` / `withdraw()` se badalta hai. Dono methods amount check
karte hain. `__str__` se account ka summary saaf print hota hai aur `__repr__` debugging ke liye hai.